In [2]:
# LIME with Your ACTUAL Unified Claims Data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import lime
import lime.lime_tabular
import warnings
warnings.filterwarnings('ignore')

print("🚀 Loading YOUR Unified Claims Data...")
print("=" * 50)

# Load YOUR actual processed data
df = pd.read_csv('../data/processed/unified_claims_v1.csv')

print(f"📊 Dataset Shape: {df.shape}")
print(f"📋 Columns: {df.columns.tolist()}")

# Explore your data
print("\n🔍 Data Overview:")
print(df.info())

print("\n🎯 Checking for Fraud Columns:")
fraud_cols = [col for col in df.columns if any(word in col.lower() for word in 
              ['fraud', 'suspect', 'abuse', 'investigation', 'flag', 'potential'])]
print(f"Fraud-related columns: {fraud_cols}")

if fraud_cols:
    for col in fraud_cols:
        print(f"\n{col} distribution:")
        print(df[col].value_counts())

print("\n📈 Numeric Columns Summary:")
print(df.describe())



🚀 Loading YOUR Unified Claims Data...
📊 Dataset Shape: (953554, 13)
📋 Columns: ['claim_id', 'patient_age', 'gender', 'hospital_id', 'admission_date', 'discharge_date', 'diagnosis_code', 'claimed_amount', 'billed_items_count', 'previous_claims_count', 'insurer_id', 'doc_missing_flag', 'is_fraud']

🔍 Data Overview:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 953554 entries, 0 to 953553
Data columns (total 13 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   claim_id               949554 non-null  object 
 1   patient_age            46613 non-null   float64
 2   gender                 253180 non-null  object 
 3   hospital_id            0 non-null       float64
 4   admission_date         717685 non-null  object 
 5   discharge_date         717449 non-null  object 
 6   diagnosis_code         731160 non-null  object 
 7   claimed_amount         744064 non-null  object 
 8   billed_items_count     0 non-null      

In [3]:
# Cell 1: Load and explore your unified data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix
import lime
import lime.lime_tabular
import warnings
warnings.filterwarnings('ignore')

print("🚀 Loading YOUR Unified Claims Data...")
print("=" * 50)

# Load YOUR actual processed data
df = pd.read_csv('../data/processed/unified_claims_v1.csv')

print(f"📊 Dataset Shape: {df.shape}")
print(f"📋 Columns: {df.columns.tolist()}")

🚀 Loading YOUR Unified Claims Data...
📊 Dataset Shape: (953554, 13)
📋 Columns: ['claim_id', 'patient_age', 'gender', 'hospital_id', 'admission_date', 'discharge_date', 'diagnosis_code', 'claimed_amount', 'billed_items_count', 'previous_claims_count', 'insurer_id', 'doc_missing_flag', 'is_fraud']


In [4]:
# Cell 2: Explore your data structure
print("\n🔍 Data Overview:")
print(df.info())

print("\n🎯 Checking for Fraud Columns:")
fraud_cols = [col for col in df.columns if any(word in col.lower() for word in 
              ['fraud', 'suspect', 'abuse', 'investigation', 'flag', 'potential'])]
print(f"Fraud-related columns: {fraud_cols}")

if fraud_cols:
    for col in fraud_cols:
        print(f"\n{col} distribution:")
        print(df[col].value_counts())
else:
    print("❌ No obvious fraud columns found - we'll need to identify the target variable")

print("\n📈 First 5 rows:")
df.head()


🔍 Data Overview:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 953554 entries, 0 to 953553
Data columns (total 13 columns):
 #   Column                 Non-Null Count   Dtype  
---  ------                 --------------   -----  
 0   claim_id               949554 non-null  object 
 1   patient_age            46613 non-null   float64
 2   gender                 253180 non-null  object 
 3   hospital_id            0 non-null       float64
 4   admission_date         717685 non-null  object 
 5   discharge_date         717449 non-null  object 
 6   diagnosis_code         731160 non-null  object 
 7   claimed_amount         744064 non-null  object 
 8   billed_items_count     0 non-null       float64
 9   previous_claims_count  0 non-null       float64
 10  insurer_id             0 non-null       float64
 11  doc_missing_flag       0 non-null       float64
 12  is_fraud               953554 non-null  int64  
dtypes: float64(6), int64(1), object(6)
memory usage: 94.6+ MB
None

🎯 Check

,claim_id,patient_age,gender,hospital_id,admission_date,discharge_date,diagnosis_code,claimed_amount,billed_items_count,previous_claims_count,insurer_id,doc_missing_flag,is_fraud
0,1,25.0,F,NaN,2025-02-11,2025-02-11,CYESIS LMP,16800.0,NaN,NaN,NaN,NaN,1
1,2,30.0,M,NaN,2025-02-13,2025-02-13,WAX IMPACTION,6300.0,NaN,NaN,NaN,NaN,1
2,3,35.0,M,NaN,2025-02-13,2025-02-13,CYESIS LMP,6160.0,NaN,NaN,NaN,NaN,1
3,4,48.0,M,NaN,2025-02-18,2025-02-18,TONSILITIS OBSTRUCTIVE SLEEP APEANA,0.0,NaN,NaN,NaN,NaN,0
4,5,58.0,F,NaN,2025-02-18,2025-02-18,REFRACTIVE ERROR,8400.0,NaN,NaN,NaN,NaN,0


In [5]:
# Cell 3: Fix data quality issues
print("🔧 Fixing Data Quality Issues...")

# Remove completely empty columns
empty_columns = ['hospital_id', 'billed_items_count', 'previous_claims_count', 'insurer_id', 'doc_missing_flag']
df_clean = df.drop(columns=empty_columns)
print(f"✅ Removed empty columns: {empty_columns}")

# Fix claimed_amount - convert to numeric, handle errors
df_clean['claimed_amount'] = pd.to_numeric(df_clean['claimed_amount'], errors='coerce')
print(f"✅ Fixed claimed_amount data type")

# Handle missing values
print("\n📊 Missing values after cleaning:")
print(df_clean.isnull().sum())

# Remove rows where critical columns are missing
initial_rows = len(df_clean)
df_clean = df_clean.dropna(subset=['patient_age', 'gender', 'admission_date', 'discharge_date', 'diagnosis_code', 'claimed_amount'])
print(f"✅ Removed rows with missing critical data: {initial_rows} -> {len(df_clean)} rows remaining")

print(f"\n📊 Clean dataset shape: {df_clean.shape}")
print(f"🎯 Fraud rate in clean data: {df_clean['is_fraud'].mean():.2%}")


🔧 Fixing Data Quality Issues...
✅ Removed empty columns: ['hospital_id', 'billed_items_count', 'previous_claims_count', 'insurer_id', 'doc_missing_flag']
✅ Fixed claimed_amount data type

📊 Missing values after cleaning:
claim_id            4000
patient_age       906941
gender            700374
admission_date    235869
discharge_date    236105
diagnosis_code    222394
claimed_amount    213422
is_fraud               0
dtype: int64
✅ Removed rows with missing critical data: 953554 -> 23367 rows remaining

📊 Clean dataset shape: (23367, 8)
🎯 Fraud rate in clean data: 34.60%


In [6]:
# Cell 4: Create features from available data
print("🔧 Creating features from available data...")

# Convert dates
df_clean['admission_date'] = pd.to_datetime(df_clean['admission_date'])
df_clean['discharge_date'] = pd.to_datetime(df_clean['discharge_date'])

# Calculate length of stay
df_clean['length_of_stay'] = (df_clean['discharge_date'] - df_clean['admission_date']).dt.days

# Create claimed amount per day (avoid division by zero)
df_clean['claimed_per_day'] = df_clean['claimed_amount'] / (df_clean['length_of_stay'] + 1)

# Create binary flags for suspicious patterns
df_clean['high_amount_flag'] = (df_clean['claimed_amount'] > df_clean['claimed_amount'].quantile(0.95)).astype(int)
df_clean['short_stay_high_bill'] = ((df_clean['length_of_stay'] < 2) & 
                                   (df_clean['claimed_amount'] > df_clean['claimed_amount'].median())).astype(int)

# Encode categorical variables
from sklearn.preprocessing import LabelEncoder

# Gender encoding
le_gender = LabelEncoder()
df_clean['gender_encoded'] = le_gender.fit_transform(df_clean['gender'].fillna('Unknown'))

# Diagnosis encoding (use top categories)
top_diagnoses = df_clean['diagnosis_code'].value_counts().head(10).index
df_clean['diagnosis_group'] = df_clean['diagnosis_code'].apply(lambda x: x if x in top_diagnoses else 'Other')
le_diagnosis = LabelEncoder()
df_clean['diagnosis_encoded'] = le_diagnosis.fit_transform(df_clean['diagnosis_group'])

print("✅ Features created!")
print(f"Available features: {[col for col in df_clean.columns if col not in ['claim_id', 'admission_date', 'discharge_date', 'diagnosis_code', 'gender']]}")



🔧 Creating features from available data...


ValueError: unconverted data remains when parsing with format "%Y-%m-%d": " 00:00:00", at position 583. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [7]:
# Cell 4: Create features from available data
print("🔧 Creating features from available data...")

# Convert dates
df_clean['admission_date'] = pd.to_datetime(df_clean['admission_date'])
df_clean['discharge_date'] = pd.to_datetime(df_clean['discharge_date'])

# Calculate length of stay
df_clean['length_of_stay'] = (df_clean['discharge_date'] - df_clean['admission_date']).dt.days

# Create claimed amount per day (avoid division by zero)
df_clean['claimed_per_day'] = df_clean['claimed_amount'] / (df_clean['length_of_stay'] + 1)

# Create binary flags for suspicious patterns
df_clean['high_amount_flag'] = (df_clean['claimed_amount'] > df_clean['claimed_amount'].quantile(0.95)).astype(int)
df_clean['short_stay_high_bill'] = ((df_clean['length_of_stay'] < 2) & 
                                   (df_clean['claimed_amount'] > df_clean['claimed_amount'].median())).astype(int)

# Encode categorical variables
from sklearn.preprocessing import LabelEncoder

# Gender encoding
le_gender = LabelEncoder()
df_clean['gender_encoded'] = le_gender.fit_transform(df_clean['gender'].fillna('Unknown'))

# Diagnosis encoding (use top categories)
top_diagnoses = df_clean['diagnosis_code'].value_counts().head(10).index
df_clean['diagnosis_group'] = df_clean['diagnosis_code'].apply(lambda x: x if x in top_diagnoses else 'Other')
le_diagnosis = LabelEncoder()
df_clean['diagnosis_encoded'] = le_diagnosis.fit_transform(df_clean['diagnosis_group'])

print("✅ Features created!")
print(f"Available features: {[col for col in df_clean.columns if col not in ['claim_id', 'admission_date', 'discharge_date', 'diagnosis_code', 'gender']]}")


🔧 Creating features from available data...


ValueError: unconverted data remains when parsing with format "%Y-%m-%d": " 00:00:00", at position 583. You might want to try:
    - passing `format` if your strings have a consistent format;
    - passing `format='ISO8601'` if your strings are all ISO8601 but not necessarily in exactly the same format;
    - passing `format='mixed'`, and the format will be inferred for each element individually. You might want to use `dayfirst` alongside this.

In [8]:
# Cell 4: Create features from available data (FIXED VERSION)
print("🔧 Creating features from available data...")

# Convert dates - handle the time component
df_clean['admission_date'] = pd.to_datetime(df_clean['admission_date'], format='mixed', errors='coerce')
df_clean['discharge_date'] = pd.to_datetime(df_clean['discharge_date'], format='mixed', errors='coerce')

# Check for any date conversion issues
date_issues = df_clean[['admission_date', 'discharge_date']].isnull().sum()
print(f"Date conversion issues - Admission: {date_issues['admission_date']}, Discharge: {date_issues['discharge_date']}")

# Remove rows where date conversion failed
df_clean = df_clean.dropna(subset=['admission_date', 'discharge_date'])
print(f"✅ Removed rows with invalid dates: {len(df_clean)} rows remaining")

# Calculate length of stay
df_clean['length_of_stay'] = (df_clean['discharge_date'] - df_clean['admission_date']).dt.days

# Check for negative stays (data quality issue)
negative_stays = (df_clean['length_of_stay'] < 0).sum()
if negative_stays > 0:
    print(f"⚠️  Found {negative_stays} claims with negative length of stay - fixing...")
    df_clean = df_clean[df_clean['length_of_stay'] >= 0]

# Create claimed amount per day (avoid division by zero)
df_clean['claimed_per_day'] = df_clean['claimed_amount'] / (df_clean['length_of_stay'] + 1)

# Create binary flags for suspicious patterns
df_clean['high_amount_flag'] = (df_clean['claimed_amount'] > df_clean['claimed_amount'].quantile(0.95)).astype(int)
df_clean['short_stay_high_bill'] = ((df_clean['length_of_stay'] < 2) & 
                                   (df_clean['claimed_amount'] > df_clean['claimed_amount'].median())).astype(int)

# Encode categorical variables
from sklearn.preprocessing import LabelEncoder

# Gender encoding
le_gender = LabelEncoder()
df_clean['gender_encoded'] = le_gender.fit_transform(df_clean['gender'].fillna('Unknown'))

# Diagnosis encoding (use top categories)
top_diagnoses = df_clean['diagnosis_code'].value_counts().head(10).index
df_clean['diagnosis_group'] = df_clean['diagnosis_code'].apply(lambda x: x if x in top_diagnoses else 'Other')
le_diagnosis = LabelEncoder()
df_clean['diagnosis_encoded'] = le_diagnosis.fit_transform(df_clean['diagnosis_group'])

print("✅ Features created!")
print(f"Available features: {[col for col in df_clean.columns if col not in ['claim_id', 'admission_date', 'discharge_date', 'diagnosis_code', 'gender']]}")
print(f"Final dataset shape: {df_clean.shape}")
print(f"Final fraud rate: {df_clean['is_fraud'].mean():.2%}")

🔧 Creating features from available data...
Date conversion issues - Admission: 0, Discharge: 0
✅ Removed rows with invalid dates: 23367 rows remaining
⚠️  Found 8016 claims with negative length of stay - fixing...
✅ Features created!
Available features: ['patient_age', 'claimed_amount', 'is_fraud', 'length_of_stay', 'claimed_per_day', 'high_amount_flag', 'short_stay_high_bill', 'gender_encoded', 'diagnosis_group', 'diagnosis_encoded']
Final dataset shape: (15351, 15)
Final fraud rate: 24.57%


In [9]:
# Cell 5: Verify clean data and distributions
print("📊 Clean Data Overview:")
print(f"Shape: {df_clean.shape}")
print(f"Fraud rate: {df_clean['is_fraud'].mean():.2%}")
print(f"Fraud cases: {df_clean['is_fraud'].sum()}")

print("\n📈 Feature Distributions:")
print(f"Length of stay: {df_clean['length_of_stay'].describe()}")
print(f"Claim amount: ${df_clean['claimed_amount'].describe()}")
print(f"Claim per day: ${df_clean['claimed_per_day'].describe()}")

print(f"\n🔍 Suspicious Pattern Flags:")
print(f"High amount claims: {df_clean['high_amount_flag'].mean():.2%}")
print(f"Short stay + high bill: {df_clean['short_stay_high_bill'].mean():.2%}")

print(f"\n👥 Gender distribution:")
print(df_clean['gender'].value_counts())

print(f"\n🏥 Top diagnoses:")
print(df_clean['diagnosis_group'].value_counts().head())

📊 Clean Data Overview:
Shape: (15351, 15)
Fraud rate: 24.57%
Fraud cases: 3771

📈 Feature Distributions:
Length of stay: count    15351.000000
mean       387.566543
std        527.579466
min          0.000000
25%          0.000000
50%         76.000000
75%        679.000000
max       7729.000000
Name: length_of_stay, dtype: float64
Claim amount: $count     15351.000000
mean       5679.198516
std        6805.994541
min           0.000000
25%           0.000000
50%        2500.000000
75%       11074.500000
max      110000.000000
Name: claimed_amount, dtype: float64
Claim per day: $count    15351.000000
mean       158.241957
std       1574.025509
min          0.000000
25%          0.000000
50%          3.459870
75%         18.371077
max      58800.000000
Name: claimed_per_day, dtype: float64

🔍 Suspicious Pattern Flags:
High amount claims: 5.00%
Short stay + high bill: 1.17%

👥 Gender distribution:
gender
F     8535
M     6792
f       18
m        4
MF       2
Name: count, dtype: int64

🏥 

In [10]:
# Cell 6: Select features and prepare for modeling
print("🎯 Selecting features for fraud detection...")

# First, let's clean up the gender data (fix the casing issues)
df_clean['gender_clean'] = df_clean['gender'].str.upper().str.strip()
df_clean['gender_clean'] = df_clean['gender_clean'].replace({'MF': 'M'})  # Handle rare cases

# Re-encode gender with clean data
from sklearn.preprocessing import LabelEncoder
le_gender = LabelEncoder()
df_clean['gender_encoded'] = le_gender.fit_transform(df_clean['gender_clean'])

feature_columns = [
    'patient_age', 'claimed_amount', 'length_of_stay', 'claimed_per_day',
    'high_amount_flag', 'short_stay_high_bill', 'gender_encoded', 'diagnosis_encoded'
]

X = df_clean[feature_columns].fillna(0)
y = df_clean['is_fraud']

print(f"📊 Feature matrix: {X.shape}")
print(f"🎯 Target: {y.shape}, Fraud rate: {y.mean():.2%}")

print(f"\n📋 Features for modeling:")
for i, feature in enumerate(feature_columns, 1):
    print(f"   {i}. {feature}")

# Train-test split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.3, 
    random_state=42, 
    stratify=y
)

print(f"\n📚 Training set: {X_train.shape}, Fraud rate: {y_train.mean():.2%}")
print(f"🧪 Test set: {X_test.shape}, Fraud rate: {y_test.mean():.2%}")

🎯 Selecting features for fraud detection...
📊 Feature matrix: (15351, 8)
🎯 Target: (15351,), Fraud rate: 24.57%

📋 Features for modeling:
   1. patient_age
   2. claimed_amount
   3. length_of_stay
   4. claimed_per_day
   5. high_amount_flag
   6. short_stay_high_bill
   7. gender_encoded
   8. diagnosis_encoded

📚 Training set: (10745, 8), Fraud rate: 24.57%
🧪 Test set: (4606, 8), Fraud rate: 24.55%


In [11]:
# Cell 7: Train the fraud detection model
print("🤖 Training Fraud Detection Model...")

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Train Random Forest
model = RandomForestClassifier(
    n_estimators=100,
    max_depth=10,
    class_weight='balanced',  # Important for imbalanced data
    random_state=42
)

model.fit(X_train, y_train)

# Evaluate model
train_predictions = model.predict(X_train)
test_predictions = model.predict(X_test)

train_accuracy = model.score(X_train, y_train)
test_accuracy = model.score(X_test, y_test)

print(f"📊 Model Performance:")
print(f"   Training Accuracy: {train_accuracy:.3f}")
print(f"   Test Accuracy: {test_accuracy:.3f}")

# Feature importance
feature_importance = pd.DataFrame({
    'feature': feature_columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

print(f"\n🔝 Most Important Features:")
print(feature_importance.head(10))

# Show confusion matrix
print(f"\n🎯 Test Set Performance:")
print(classification_report(y_test, test_predictions, target_names=['Genuine', 'Fraud']))

🤖 Training Fraud Detection Model...
📊 Model Performance:
   Training Accuracy: 0.759
   Test Accuracy: 0.765

🔝 Most Important Features:
                feature  importance
1        claimed_amount    0.405929
3       claimed_per_day    0.241234
2        length_of_stay    0.233444
0           patient_age    0.075053
6        gender_encoded    0.018943
7     diagnosis_encoded    0.013165
5  short_stay_high_bill    0.006699
4      high_amount_flag    0.005535

🎯 Test Set Performance:
              precision    recall  f1-score   support

     Genuine       0.97      0.71      0.82      3475
       Fraud       0.51      0.93      0.66      1131

    accuracy                           0.77      4606
   macro avg       0.74      0.82      0.74      4606
weighted avg       0.86      0.77      0.78      4606



In [12]:
# Cell 8: Implement LIME Explanations
print("🕵️ Implementing LIME Explanations...")

# Initialize LIME Explainer
explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=feature_columns,
    class_names=['Genuine', 'Fraud'],
    mode='classification',
    random_state=42
)

print("✅ LIME Explainer Ready!")

# Test LIME on a few cases
def explain_fraud_prediction(instance_idx, num_features=5):
    """Generate LIME explanation for a specific claim"""
    instance = X_test.iloc[instance_idx:instance_idx+1].values[0]
    true_label = y_test.iloc[instance_idx]
    prediction = model.predict_proba([instance])[0]
    
    # Generate LIME explanation
    explanation = explainer.explain_instance(
        instance, 
        model.predict_proba, 
        num_features=num_features
    )
    
    print(f"\n🔍 Claim Analysis #{instance_idx}")
    print("=" * 50)
    print(f"📊 Fraud Probability: {prediction[1]:.1%}")
    print(f"🎯 Actual: {'FRAUD' if true_label else 'GENUINE'}")
    
    # Get explanation
    exp_list = explanation.as_list()
    print(f"\n🎯 Top {num_features} Contributing Factors:")
    for i, (feature, importance) in enumerate(exp_list[:num_features], 1):
        print(f"   {i}. {feature}: {importance:.3f}")
    
    return explanation

# Test on some cases
print("\n🚀 Testing LIME on Real Claims:")
print("=" * 50)

# Find some fraud and genuine cases
fraud_indices = y_test[y_test == 1].index[:2]
genuine_indices = y_test[y_test == 0].index[:1]

for idx in list(fraud_indices) + list(genuine_indices):
    explain_fraud_prediction(idx)

🕵️ Implementing LIME Explanations...
✅ LIME Explainer Ready!

🚀 Testing LIME on Real Claims:

🔍 Claim Analysis #12
📊 Fraud Probability: 75.4%
🎯 Actual: GENUINE

🎯 Top 5 Contributing Factors:
   1. claimed_amount <= 0.00: 0.299
   2. claimed_per_day <= 0.00: 0.220
   3. length_of_stay <= 0.00: 0.071
   4. gender_encoded <= 0.00: -0.071
   5. 43.00 < patient_age <= 45.23: 0.040


IndexError: index 0 is out of bounds for axis 0 with size 0

In [13]:
# Cell 8: Implement LIME Explanations (FIXED VERSION)
print("🕵️ Implementing LIME Explanations...")

# Initialize LIME Explainer
explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=feature_columns,
    class_names=['Genuine', 'Fraud'],
    mode='classification',
    random_state=42
)

print("✅ LIME Explainer Ready!")

# Test LIME on a few cases - FIXED FUNCTION
def explain_fraud_prediction(instance_idx, num_features=5):
    """Generate LIME explanation for a specific claim"""
    try:
        # Get the instance properly - use .iloc with position, not index
        instance = X_test.iloc[instance_idx:instance_idx+1]
        if len(instance) == 0:
            print(f"❌ No data at position {instance_idx}")
            return None
            
        instance_values = instance.values[0]
        true_label = y_test.iloc[instance_idx]
        prediction = model.predict_proba([instance_values])[0]
        
        # Generate LIME explanation
        explanation = explainer.explain_instance(
            instance_values, 
            model.predict_proba, 
            num_features=num_features
        )
        
        print(f"\n🔍 Claim Analysis #{instance_idx}")
        print("=" * 50)
        print(f"📊 Fraud Probability: {prediction[1]:.1%}")
        print(f"🎯 Actual: {'FRAUD' if true_label else 'GENUINE'}")
        
        # Get explanation
        exp_list = explanation.as_list()
        print(f"\n🎯 Top {num_features} Contributing Factors:")
        for i, (feature, importance) in enumerate(exp_list[:num_features], 1):
            print(f"   {i}. {feature}: {importance:.3f}")
        
        return explanation
        
    except Exception as e:
        print(f"❌ Error analyzing claim #{instance_idx}: {e}")
        return None

# Test on some cases - USE POSITIONS, NOT INDEXES
print("\n🚀 Testing LIME on Real Claims:")
print("=" * 50)

# Use actual positions in the test set (0, 1, 2, etc.)
test_positions = [0, 1, 2, 10, 20]  # Just use first few positions

for pos in test_positions:
    if pos < len(X_test):  # Make sure position exists
        explain_fraud_prediction(pos)
    else:
        print(f"⚠️ Position {pos} beyond test set size")

🕵️ Implementing LIME Explanations...
✅ LIME Explainer Ready!

🚀 Testing LIME on Real Claims:

🔍 Claim Analysis #0
📊 Fraud Probability: 1.2%
🎯 Actual: GENUINE

🎯 Top 5 Contributing Factors:
   1. claimed_per_day > 18.21: -0.156
   2. claimed_amount > 11006.00: -0.147
   3. gender_encoded <= 0.00: -0.076
   4. 77.00 < length_of_stay <= 678.00: -0.033
   5. patient_age <= 43.00: -0.015

🔍 Claim Analysis #1
📊 Fraud Probability: 10.1%
🎯 Actual: GENUINE

🎯 Top 5 Contributing Factors:
   1. claimed_per_day > 18.21: -0.150
   2. claimed_amount > 11006.00: -0.142
   3. 0.00 < gender_encoded <= 1.00: 0.075
   4. 77.00 < length_of_stay <= 678.00: -0.043
   5. patient_age <= 43.00: -0.030

🔍 Claim Analysis #2
📊 Fraud Probability: 74.6%
🎯 Actual: GENUINE

🎯 Top 5 Contributing Factors:
   1. claimed_amount <= 0.00: 0.297
   2. claimed_per_day <= 0.00: 0.215
   3. 0.00 < gender_encoded <= 1.00: 0.071
   4. length_of_stay <= 0.00: 0.069
   5. 43.00 < patient_age <= 45.23: 0.042

🔍 Claim Analysis #10
📊

In [14]:
# Cell 9: Create Business-Friendly Explanations
print("\n💼 Creating Business-Friendly Explanations...")

def create_business_explanation(explanation, feature_names):
    """Convert LIME output to insurance business language"""
    exp_list = explanation.as_list()
    
    reasons = []
    for feature, importance in exp_list[:3]:  # Top 3 reasons
        # Parse LIME feature string
        if ' <= ' in feature:
            parts = feature.split(' <= ')
            feature_name = parts[0]
            threshold = float(parts[1])
            
            # Map feature names to business terms
            feature_map = {
                'claimed_amount': 'Claim Amount',
                'claimed_per_day': 'Daily Claim Rate', 
                'length_of_stay': 'Hospital Stay Duration',
                'patient_age': 'Patient Age',
                'gender_encoded': 'Patient Gender',
                'diagnosis_encoded': 'Diagnosis Type'
            }
            
            clean_name = feature_map.get(feature_name, feature_name.replace('_', ' ').title())
            
            if importance > 0:
                if 'amount' in feature_name or 'claimed' in feature_name:
                    reason = f"Low {clean_name} (${threshold:,.0f})"
                else:
                    reason = f"Low {clean_name} ({threshold:.0f} days)"
            else:
                if 'amount' in feature_name or 'claimed' in feature_name:
                    reason = f"High {clean_name} (above ${threshold:,.0f})"
                else:
                    reason = f"High {clean_name} (above {threshold:.0f} days)"
                
        elif ' > ' in feature:
            parts = feature.split(' > ')
            feature_name = parts[0]
            threshold = float(parts[1])
            
            feature_map = {
                'claimed_amount': 'Claim Amount',
                'claimed_per_day': 'Daily Claim Rate',
                'length_of_stay': 'Hospital Stay Duration', 
                'patient_age': 'Patient Age',
                'gender_encoded': 'Patient Gender',
                'diagnosis_encoded': 'Diagnosis Type'
            }
            
            clean_name = feature_map.get(feature_name, feature_name.replace('_', ' ').title())
            
            if importance > 0:
                if 'amount' in feature_name or 'claimed' in feature_name:
                    reason = f"High {clean_name} (${threshold:,.0f})"
                else:
                    reason = f"High {clean_name} ({threshold:.0f} days)"
            else:
                if 'amount' in feature_name or 'claimed' in feature_name:
                    reason = f"Low {clean_name} (below ${threshold:,.0f})"
                else:
                    reason = f"Low {clean_name} (below {threshold:.0f} days)"
        else:
            # Simple feature
            feature_clean = feature.replace('_', ' ').title()
            if importance > 0:
                reason = f"High {feature_clean}"
            else:
                reason = f"Low {feature_clean}"
        
        reasons.append(reason)
    
    return reasons

# Test business explanations with fixed positions
print("📋 Business Explanations for Claims:")
print("=" * 50)

# Test on interesting cases we saw
test_positions = [0, 2, 10]  # Low risk, high risk, medium risk

for pos in test_positions:
    if pos < len(X_test):
        instance = X_test.iloc[pos:pos+1].values[0]
        true_label = y_test.iloc[pos]
        fraud_prob = model.predict_proba([instance])[0][1]
        
        explanation = explainer.explain_instance(instance, model.predict_proba, num_features=5)
        
        if explanation:
            business_reasons = create_business_explanation(explanation, feature_columns)
            
            print(f"\n📄 Claim #{pos}")
            print(f"   Risk Score: {fraud_prob:.1%}")
            print(f"   Status: {'🚨 FRAUD' if true_label else '✅ GENUINE'}")
            print(f"   Key Risk Factors:")
            for i, reason in enumerate(business_reasons, 1):
                print(f"     {i}. {reason}")
            
            # Add recommendation
            if fraud_prob > 0.7:
                print(f"   🚨 Recommendation: Flag for immediate review")
                print(f"   🔍 Action: Verify treatment necessity and billing accuracy")
            elif fraud_prob > 0.3:
                print(f"   ⚠️  Recommendation: Additional verification needed")
                print(f"   🔍 Action: Request additional documentation")
            else:
                print(f"   ✅ Recommendation: Approve automatically")
                print(f"   📝 Action: Standard processing")
                


💼 Creating Business-Friendly Explanations...
📋 Business Explanations for Claims:

📄 Claim #0
   Risk Score: 1.2%
   Status: ✅ GENUINE
   Key Risk Factors:
     1. Low Daily Claim Rate (below $18)
     2. Low Claim Amount (below $11,006)
     3. High Patient Gender (above 0 days)
   ✅ Recommendation: Approve automatically
   📝 Action: Standard processing

📄 Claim #2
   Risk Score: 74.6%
   Status: ✅ GENUINE
   Key Risk Factors:
     1. Low Claim Amount ($0)
     2. Low Daily Claim Rate ($0)
     3. Low 0.00 < Gender Encoded (1 days)
   🚨 Recommendation: Flag for immediate review
   🔍 Action: Verify treatment necessity and billing accuracy

📄 Claim #10
   Risk Score: 11.6%
   Status: ✅ GENUINE
   Key Risk Factors:
     1. High 2458.00 < Claimed Amount (above $11,006)
     2. Low Daily Claim Rate (below $18)
     3. Low 0.00 < Gender Encoded (1 days)
   ✅ Recommendation: Approve automatically
   📝 Action: Standard processing


In [15]:
# Cell 10: Save model and explainer for production
print("\n💾 Saving Model and LIME Explainer...")

import pickle
import joblib
from datetime import datetime

# Save the model with metadata
model_data = {
    'model': model,
    'feature_names': feature_columns,
    'feature_encoder': {
        'gender': le_gender,
        'diagnosis': le_diagnosis
    },
    'lime_explainer': explainer,
    'training_info': {
        'dataset_size': len(df_clean),
        'fraud_rate': df_clean['is_fraud'].mean(),
        'training_date': datetime.now().strftime("%Y-%m-%d"),
        'features_used': feature_columns
    },
    'model_performance': {
        'test_accuracy': test_accuracy,
        'fraud_precision': 0.51,  # From classification report
        'fraud_recall': 0.93,
        'feature_importance': dict(zip(feature_columns, model.feature_importances_))
    }
}

# Save using joblib (better for large objects)
joblib.dump(model_data, '../models/fraud_detection_model_with_lime.joblib')

print("✅ Model saved: ../models/fraud_detection_model_with_lime.joblib")

# Also save a simple version for quick testing
simple_model_data = {
    'model': model,
    'feature_names': feature_columns,
    'lime_explainer': explainer
}
joblib.dump(simple_model_data, '../models/fraud_model_simple.joblib')

print("✅ Simple model saved: ../models/fraud_model_simple.joblib")


💾 Saving Model and LIME Explainer...


PicklingError: Can't pickle <function BaseDiscretizer.__init__.<locals>.<lambda> at 0x0000029C0AA7FC40>: it's not found as lime.discretize.BaseDiscretizer.__init__.<locals>.<lambda>

In [16]:
# Cell 10: Save model and explainer for production (FIXED VERSION)
print("\n💾 Saving Model and LIME Components...")

import pickle
import joblib
from datetime import datetime

# Create models directory if it doesn't exist
import os
os.makedirs('../models', exist_ok=True)

# OPTION 1: Save just the model and feature info (ALWAYS WORKS)
print("💾 Option 1: Saving core model (pickle-friendly)...")

core_model_data = {
    'model': model,
    'feature_names': feature_columns,
    'feature_encoder': {
        'gender': le_gender,
        'diagnosis': le_diagnosis
    },
    'training_info': {
        'dataset_size': len(df_clean),
        'fraud_rate': df_clean['is_fraud'].mean(),
        'training_date': datetime.now().strftime("%Y-%m-%d"),
        'features_used': feature_columns
    },
    'model_performance': {
        'test_accuracy': test_accuracy,
        'fraud_precision': 0.51,
        'fraud_recall': 0.93,
        'feature_importance': dict(zip(feature_columns, model.feature_importances_))
    },
    'lime_training_data': X_train.values,  # Save training data to recreate LIME
    'lime_feature_names': feature_columns
}

# Save core model (this should work)
joblib.dump(core_model_data, '../models/fraud_detection_core_model.joblib')
print("✅ Core model saved: ../models/fraud_detection_core_model.joblib")

# OPTION 2: Try to save LIME explainer separately (might fail but worth trying)
print("\n💾 Option 2: Trying to save LIME explainer...")
try:
    # Save LIME explainer separately
    lime_data = {
        'explainer': explainer,
        'feature_names': feature_columns
    }
    joblib.dump(lime_data, '../models/lime_explainer.joblib')
    print("✅ LIME explainer saved: ../models/lime_explainer.joblib")
except Exception as e:
    print(f"⚠️  Could not save LIME explainer: {e}")
    print("💡 Don't worry - we can recreate LIME explainer from training data")

# OPTION 3: Save simple model for API (most reliable)
print("\n💾 Option 3: Saving simple model for API...")
simple_model = {
    'model': model,
    'feature_names': feature_columns,
    'encoders': {
        'gender': le_gender,
        'diagnosis': le_diagnosis
    }
}

joblib.dump(simple_model, '../models/fraud_model_api_ready.joblib')
print("✅ API-ready model saved: ../models/fraud_model_api_ready.joblib")

print("\n📁 Files created in models/ directory:")
models_dir = '../models'
for file in os.listdir(models_dir):
    if file.endswith('.joblib'):
        file_path = os.path.join(models_dir, file)
        size_mb = os.path.getsize(file_path) / (1024 * 1024)
        print(f"   📄 {file} ({size_mb:.1f} MB)")


💾 Saving Model and LIME Components...
💾 Option 1: Saving core model (pickle-friendly)...
✅ Core model saved: ../models/fraud_detection_core_model.joblib

💾 Option 2: Trying to save LIME explainer...
⚠️  Could not save LIME explainer: Can't pickle <function BaseDiscretizer.__init__.<locals>.<lambda> at 0x0000029C0AA7FC40>: it's not found as lime.discretize.BaseDiscretizer.__init__.<locals>.<lambda>
💡 Don't worry - we can recreate LIME explainer from training data

💾 Option 3: Saving simple model for API...
✅ API-ready model saved: ../models/fraud_model_api_ready.joblib

📁 Files created in models/ directory:
   📄 fraud_detection_core_model.joblib (2.6 MB)
   📄 fraud_detection_model_with_lime.joblib (1.9 MB)
   📄 fraud_model_api_ready.joblib (1.9 MB)
   📄 lime_explainer.joblib (0.0 MB)


In [17]:
# Cell 11: Create LIME recreation function for production
print("\n🔧 Creating LIME Recreation Function...")

def create_lime_explainer(model, training_data, feature_names):
    """Recreate LIME explainer in production (solves pickling issues)"""
    import lime.lime_tabular
    
    explainer = lime.lime_tabular.LimeTabularExplainer(
        training_data=training_data,
        feature_names=feature_names,
        class_names=['Genuine', 'Fraud'],
        mode='classification',
        random_state=42
    )
    return explainer

# Test the recreation function
print("🧪 Testing LIME recreation...")
recreated_explainer = create_lime_explainer(model, X_train.values, feature_columns)

# Test that it works
test_instance = X_test.iloc[0:1].values[0]
test_explanation = recreated_explainer.explain_instance(test_instance, model.predict_proba, num_features=3)

print("✅ LIME recreation successful!")
print(f"Test explanation: {[item[0] for item in test_explanation.as_list()[:2]]}")


🔧 Creating LIME Recreation Function...
🧪 Testing LIME recreation...
✅ LIME recreation successful!
Test explanation: ['claimed_per_day > 18.21', 'claimed_amount > 11006.00']
